In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os 
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



In [ ]:
from  langchain_core.messages import AIMessage,HumanMessage
from pprint import pprint
messages = [AIMessage(content = f"Please tell me how can i help you?",name = "LLM Model ")]
messages.append(HumanMessage(content="I want to learn coding",name="Divyanshu"))
messages.append(AIMessage(content = "Which programming language you want to learn? ",name = "Divyanshu"))

for message in messages:
    message.pretty_print()

In [ ]:
from langchain_groq import ChatGroq 
llm = ChatGroq(model = "openai/gpt-oss-20b")
result = llm.invoke(messages)
result.pretty_print()

In [ ]:
def add(a:int,b:int)->int:
    """
    Add a and b 
    Args:
       a(int): first int
       b(int): second int 
       Returns: 
          int
    """
    return a+b

In [ ]:
## Binding tool with llm 
llm_with_tools = llm.bind_tools([add])
result = llm_with_tools.invoke([HumanMessage(content = f"What is 2 plus 2",name = "Divyanshu")])
result.pretty_print()

In [ ]:
# Using messages as state 
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing  import Annotated
class State(TypedDict):
    message:Annotated[list[AnyMessage],add_messages]


In [ ]:
# Reducer with add_messages 
initial_messages = [AIMessage(content = "How can i help you",author = "DIVYANSHU")]
initial_messages.append(HumanMessage(content = "I want to learn Coding",author = "LLM"))
ai_message = AIMessage(content = "What programming language you want to learn?",author = "LLM")
add_messages(initial_messages,ai_message)


In [ ]:
def llm_tool(state:State):
    return {"messages":{llm_with_tools.invoke(state["messages"])}}

In [ ]:
from IPython.display import Image,display
from langgraph.graph import StateGraph,START,END
builder = StateGraph(State)
builder.add_node("llm_tool",llm_tool)
builder.add_edge(START,"llm_tool")
builder.add_edge("llm_tool",END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

